# Pipeline Entity Drift — End to End Demo

This notebook walks through the full pipeline:
1. Pull real company data from SEC EDGAR
2. Build monthly entity snapshots
3. Run the hybrid scoring algorithm
4. Load the temporal graph into Neo4j AuraDB

**Requirements:** `pip install pandas numpy requests neo4j tqdm`

**Before running:** Update `HEADERS` in `edgar_pull.py` with your email (SEC requirement).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import difflib
import re
import random
import json
from pathlib import Path

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('All imports ready')

## 2. Pull Data from SEC EDGAR

In [ ]:
# Pulls 300 companies with documented name changes from SEC EDGAR
# Outputs: data/entities_raw.csv, data/revenue_snapshots.csv
%run edgar_pull.py

## 3. Build Monthly Entity Snapshots

In [ ]:
# Load raw EDGAR data
entities = pd.read_csv(DATA_DIR / 'entities_raw.csv')
revenue  = pd.read_csv(DATA_DIR / 'revenue_snapshots.csv', dtype={'cik': str})

# Normalize CIK to 10-digit zero-padded format
entities['cik'] = entities['cik'].astype(str).str.zfill(10)
revenue['cik']  = revenue['cik'].astype(str).str.zfill(10)

# Convert period to YYYY-MM format
revenue['month'] = pd.to_datetime(revenue['period']).dt.to_period('M').astype(str)

# Deduplicate — take max revenue per company per month
revenue_monthly = (
    revenue.groupby(['cik', 'month'])['revenue_usd']
    .max()
    .reset_index()
)

print(f'Revenue records after dedup : {len(revenue_monthly)}')
print(f'Date range: {revenue_monthly["month"].min()} → {revenue_monthly["month"].max()}')
revenue_monthly.head()

In [ ]:
# Verify name history dates are populated
for _, entity in entities.head(5).iterrows():
    nh = json.loads(entity['name_history'])
    former = [n for n in nh if not n['is_current']]
    if former:
        print(f"\n{entity['current_name']}")
        for n in former:
            print(f"  {n['name']} → changed: {n['date_changed']}")

In [ ]:
# Build one EntitySnapshot row per company per month
# Assigns the correct historical name for each month
# Flags the exact month a rename occurred as drift_type='rename'

rows = []
snapshot_id = 1

for _, entity in entities.iterrows():
    cik          = entity['cik']
    current_name = entity['current_name']
    name_history = json.loads(entity['name_history'])

    former = [n for n in name_history if not n['is_current'] and n['date_changed']]
    former = sorted(former, key=lambda x: x['date_changed'])

    rev = revenue_monthly[revenue_monthly['cik'] == cik].sort_values('month')
    if rev.empty:
        continue

    for _, rev_row in rev.iterrows():
        month      = rev_row['month']
        name       = current_name
        drift_type = 'none'

        for i, record in enumerate(former):
            change_month = record['date_changed']
            if month < change_month:
                name = record['name']
                break
            elif month == change_month:
                drift_type = 'rename'
                name = current_name if i == len(former) - 1 else former[i + 1]['name']
                break

        rows.append({
            'snapshot_id':   f'S{snapshot_id:05d}',
            'cik':           cik,
            'month':         month,
            'entity_name':   name,
            'revenue_usd':   rev_row['revenue_usd'],
            'drift_type':    drift_type,
            'canonical_cik': cik,
        })
        snapshot_id += 1

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / 'entity_snapshots.csv', index=False)

print(f'Total snapshots : {len(df)}')
print(f'Unique entities : {df["cik"].nunique()}')
print(f'Drift events    : {df[df["drift_type"] == "rename"].shape[0]}')
print(f'Month range     : {df["month"].min()} → {df["month"].max()}')
df[df['drift_type'] == 'rename'][['cik','month','entity_name','drift_type']].head(15)

## 4. Run Hybrid Scoring Algorithm

In [ ]:
# Scores 9,137 consecutive monthly pairs
# Outputs: data/resolution_scores.csv
%run resolution_algorithm.py

## 5. Load into Neo4j AuraDB

In [ ]:
%pip install neo4j

In [ ]:
from neo4j import GraphDatabase

# Set your AuraDB credentials here
# Get a free instance at https://console.neo4j.io
URI      = 'neo4j+s://xxxx.databases.neo4j.io'  # replace with your URI
USER     = 'neo4j'
PASSWORD = 'your-password'                        # replace with your password

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print('Connected!')

In [ ]:
# Load data
snapshots = pd.read_csv(DATA_DIR / 'entity_snapshots.csv', dtype={'cik': str})
scores    = pd.read_csv(DATA_DIR / 'resolution_scores.csv', dtype={'cik': str})

snapshots['cik'] = snapshots['cik'].str.zfill(10)
scores['cik']    = scores['cik'].str.zfill(10)

print(f'Snapshots : {len(snapshots)}')
print(f'Scores    : {len(scores)}')

In [ ]:
# Create schema
with driver.session() as session:
    session.run('CREATE CONSTRAINT snapshot_id IF NOT EXISTS FOR (s:EntitySnapshot) REQUIRE s.snapshot_id IS UNIQUE')
    session.run('CREATE CONSTRAINT canonical_cik IF NOT EXISTS FOR (c:CanonicalEntity) REQUIRE c.cik IS UNIQUE')
    session.run('CREATE CONSTRAINT drift_event_id IF NOT EXISTS FOR (d:DriftEvent) REQUIRE d.event_id IS UNIQUE')
    session.run('CREATE INDEX snapshot_month IF NOT EXISTS FOR (s:EntitySnapshot) ON (s.month)')
    session.run('CREATE INDEX snapshot_cik IF NOT EXISTS FOR (s:EntitySnapshot) ON (s.cik)')
print('Schema created')

In [ ]:
# Load CanonicalEntity nodes
canonical = (
    snapshots.groupby('cik')
    .agg(current_name=('entity_name','last'), first_seen=('month','min'), last_seen=('month','max'))
    .reset_index()
)

with driver.session() as session:
    for _, row in canonical.iterrows():
        session.run("""
            MERGE (c:CanonicalEntity {cik: $cik})
            SET c.current_name = $current_name,
                c.first_seen   = $first_seen,
                c.last_seen    = $last_seen
        """, row.to_dict())

print(f'Loaded {len(canonical)} CanonicalEntity nodes')

In [ ]:
# Load EntitySnapshot nodes
from tqdm import tqdm

with driver.session() as session:
    for _, row in tqdm(snapshots.iterrows(), total=len(snapshots)):
        session.run("""
            MERGE (s:EntitySnapshot {snapshot_id: $snapshot_id})
            SET s.cik         = $cik,
                s.month       = $month,
                s.entity_name = $entity_name,
                s.revenue_usd = $revenue_usd,
                s.drift_type  = $drift_type
        """, {
            'snapshot_id': row['snapshot_id'],
            'cik':         row['cik'],
            'month':       row['month'],
            'entity_name': row['entity_name'],
            'revenue_usd': float(row['revenue_usd']),
            'drift_type':  row['drift_type'],
        })

print(f'Loaded {len(snapshots)} EntitySnapshot nodes')

In [ ]:
# Link snapshots to canonical entities
with driver.session() as session:
    for _, row in tqdm(snapshots.iterrows(), total=len(snapshots)):
        session.run("""
            MATCH (s:EntitySnapshot {snapshot_id: $snapshot_id})
            MATCH (c:CanonicalEntity {cik: $cik})
            MERGE (s)-[:SNAPSHOT_OF]->(c)
        """, {'snapshot_id': row['snapshot_id'], 'cik': row['cik']})

print('SNAPSHOT_OF relationships created')

In [ ]:
# Load resolution relationships
lookup = snapshots.set_index(['cik', 'month'])['snapshot_id'].to_dict()

with driver.session() as session:
    for _, row in tqdm(scores.iterrows(), total=len(scores)):
        cik      = row['cik'].zfill(10)
        sid_from = lookup.get((cik, row['month_from']))
        sid_to   = lookup.get((cik, row['month_to']))

        if not sid_from or not sid_to:
            continue

        session.run("""
            MATCH (a:EntitySnapshot {snapshot_id: $sid_from})
            MATCH (b:EntitySnapshot {snapshot_id: $sid_to})
            MERGE (b)-[:PRECEDED_BY]->(a)
        """, {'sid_from': sid_from, 'sid_to': sid_to})

        if row['outcome'] in ('AUTO_RESOLVE', 'REVIEW'):
            session.run("""
                MATCH (a:EntitySnapshot {snapshot_id: $sid_from})
                MATCH (b:EntitySnapshot {snapshot_id: $sid_to})
                MERGE (a)-[r:RESOLVES_TO]->(b)
                SET r.hybrid_score = $hybrid_score, r.outcome = $outcome
            """, {'sid_from': sid_from, 'sid_to': sid_to,
                  'hybrid_score': float(row['hybrid_score']), 'outcome': row['outcome']})

        if row['conflict']:
            session.run("""
                MATCH (a:EntitySnapshot {snapshot_id: $sid_from})
                MATCH (b:EntitySnapshot {snapshot_id: $sid_to})
                MERGE (a)-[r:CONFLICTS_WITH]->(b)
                SET r.conflict_type = $conflict_type,
                    r.hybrid_score  = $hybrid_score,
                    r.name_score    = $name_score,
                    r.revenue_score = $revenue_score
            """, {'sid_from': sid_from, 'sid_to': sid_to,
                  'conflict_type':  row['conflict_type'],
                  'hybrid_score':   float(row['hybrid_score']),
                  'name_score':     float(row['name_score']),
                  'revenue_score':  float(row['revenue_score'])})

print('Resolution relationships created')

In [ ]:
# Verify graph loaded correctly
with driver.session() as session:
    result = session.run('MATCH (n) RETURN labels(n) AS label, count(n) AS count')
    for record in result:
        print(record['label'], ':', record['count'])

## 6. Run Demo Queries

Open `demo_queries.cypher` in the Neo4j browser and run the queries in order.

Or run them here via the Python driver:

In [ ]:
# Demo Query: Find all NAME_CHANGED_REVENUE_STABLE drift events
with driver.session() as session:
    result = session.run("""
        MATCH (a:EntitySnapshot)-[r:CONFLICTS_WITH]->(b:EntitySnapshot)
        WHERE r.conflict_type = 'NAME_CHANGED_REVENUE_STABLE'
        RETURN a.entity_name AS name_before,
               b.entity_name AS name_after,
               a.month       AS month,
               r.name_score  AS name_similarity,
               r.revenue_score AS revenue_continuity,
               r.hybrid_score  AS combined_score
        ORDER BY r.name_score ASC
    """)
    df_drift = pd.DataFrame([dict(r) for r in result])

df_drift